<a href="https://colab.research.google.com/github/801-Hillside-Terrace/SMART-2026/blob/main/week1/Week1_Pytorch_Functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [166]:
#imports
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import csv

#load data
url = 'https://raw.githubusercontent.com/801-Hillside-Terrace/SMART-2026/main/week1/Concrete_Data%20-%20Sheet1.csv'
concrete_data = pd.read_csv(url)
concrete_data.columns

Index(['Cement (component 1)(kg in a m^3 mixture)',
       'Blast Furnace Slag (component 2)(kg in a m^3 mixture)',
       'Fly Ash (component 3)(kg in a m^3 mixture)',
       'Water  (component 4)(kg in a m^3 mixture)',
       'Superplasticizer (component 5)(kg in a m^3 mixture)',
       'Coarse Aggregate  (component 6)(kg in a m^3 mixture)',
       'Fine Aggregate (component 7)(kg in a m^3 mixture)', 'Age (day)',
       'Concrete compressive strength(MPa, megapascals) '],
      dtype='object')

In [167]:
# working with tensors:

# convert from dataframe to numpy arrays (pandas to numpy)
y_np = concrete_data['Concrete compressive strength(MPa, megapascals) '].to_numpy()
X_np = concrete_data.drop(columns = ['Concrete compressive strength(MPa, megapascals) ']).to_numpy()

# convert to tensors/create tensors/change shapes
X_tensor = torch.tensor(X_np, dtype = torch.float32) # float32 is preferred for being accurate enough but more memory efficient than float64 (4 bytes vs 8 bytes)
y_tensor = torch.tensor(y_np, dtype = torch.float32).view(-1, 1) # ensures that its second dimension (# of columns) exists and is 1, the -1 means infer the number of rows
y_tensor = torch.tensor(y_np, dtype = torch.float32).reshape(-1, 1) # alternative to above
y_tensor = torch.tensor(y_np, dtype = torch.float32).unsqueeze(1) # alternative to above, adds new dimension at second shape index (adds column dimension)
Xy_tensor = torch.cat([X_tensor, y_tensor], dim = 1) # combine X and y (add y as new column)
X_tensor_remade = torch.cat([X_tensor[:5], X_tensor[5:]], dim = 0) # can concatenate rows as well
print(torch.equal(X_tensor_remade, X_tensor))
m = torch.tensor([[1,2], [3,4]], dtype = torch.float32) # manual tensor creation
print('m: \n',m)
m2 = m.reshape(1,4) # converts from 2x2 to 1x4
print('m2: \n',m2)
m3 = m2.squeeze(0) # converts from 1x4 to 4 (drops first dimension)
print('m3: \n',m3)
m4 = m.unsqueeze(0) # adds a third dimension
print('m4: \n',m4)
print('m4 shape: ', m4.shape) # first dimension given in shape is the "outermost" for tensors with dimension above 2, last 2 dimensions are still essentially rows and columns
m5 = m4.permute(2,1,0) # change shape from 1x2x2 to 2x2x1
print('m5 shape: ', m5.shape)
m_np = m.cpu().numpy() # converts pytorch tensor to numpy array (need to move to cpu first if using cuda)
m_torch = torch.from_numpy(m_np) # converts numpy array to pytorch tensor

# check shapes
print('X_tensor shape: ', X_tensor.shape)
print('y_tensor shape: ', y_tensor.shape)
print('X_tensor # of elements: ', X_tensor.numel())

# indexing (similar to iloc and numpy arrays):
X_tensor[0] # first row
m3[0] # first element of outermost dimension
X_tensor[0:2] # first two rows
X_tensor[:,0] # all rows, first column
X_tensor[:,0:2] # all rows, first two columns
torch.arange(5) # creates a tensor of 0, 1, 2, 3, 4

# misc
print('3x2x1 tensor of zeros: \n', torch.zeros((3,2,1))) # creates a 3x2x1 tensor of zeros (tensors can have more than 2 dimensions)
print('1x5 tensor of ones: \n', torch.ones((1,5))) # creates a 1x5 tensor of ones
u = torch.rand((1,5)) # creates 1x5 tensor of 5 random uniform(0,1)
z = torch.randn((10,1)) # creates 10x1 tensor of random standard normals
random_indices = torch.randperm(10) # creates a random ordering of 0 through 9 (first 10 indices)
random_eight = z[random_indices[:8]] # 8 shuffled values from z
random_two = z[random_indices[8:]] # remaining 2 shuffled values from z
z.device # check device the tensor is stored on
device = "cuda" if torch.cuda.is_available() else "cpu" # set device to cuda if cuda is available
z.to(device) # move z to device

# math
z = z[:5].reshape(-1, 5) # select first 5 rows of z, convert from 5x1 to 1x5
q = z + u # add
e = z * u # multiply
l = torch.log(z) # natural log
o = torch.exp(l) # exponential function
v = torch.matmul(z,u.permute(1,0)) # matrix multiplication (transposed u with permute)
v = z @ u.permute(1,0) # matrix multiplication (transposed u with permute)
c = torch.dot(z.squeeze(0),u.squeeze(0)) # squeeze off row dimension (drops dimension at 0th index) (make them both have a single dimension of 5) and take dot product (only works if they are one dimensional vectors)

# auto gradients
x = torch.tensor(7.5, dtype = torch.float32, requires_grad = True) # create tensor with gradient tracking (False by default)
y = torch.tensor(5, dtype = torch.float32, requires_grad = False) # create tensor without gradient tracking
y.requires_grad_(True) # set y to have gradient tracking
z = torch.randn((1,5), requires_grad = True) # we can do this for rand and randn too
fx = x**2 + 5*x # "function" of x; y = x^2+5x
fx.backward() # computes gradient with respect to all tensors with gradient tracking involved in the function (in thise case, makes it 2x+5)
print(x.grad) # see gradient of fx wrt x (should be 20: 2x+5 where x = 7.5)
x = x.detach() # remove gradient tracking (must be done before converting to numpy arrays etc as well)

True
m: 
 tensor([[1., 2.],
        [3., 4.]])
m2: 
 tensor([[1., 2., 3., 4.]])
m3: 
 tensor([1., 2., 3., 4.])
m4: 
 tensor([[[1., 2.],
         [3., 4.]]])
m4 shape:  torch.Size([1, 2, 2])
m5 shape:  torch.Size([2, 2, 1])
X_tensor shape:  torch.Size([1030, 8])
y_tensor shape:  torch.Size([1030, 1])
X_tensor # of elements:  8240
3x2x1 tensor of zeros: 
 tensor([[[0.],
         [0.]],

        [[0.],
         [0.]],

        [[0.],
         [0.]]])
1x5 tensor of ones: 
 tensor([[1., 1., 1., 1., 1.]])
tensor(20.)


In [168]:
# practice:
# 1. Create a tensor called A with shape 5 (not 5x1 or 1x5) and elements 0,2,4,6,8

# 2. Make tensor A have shape 1x5

# 3. Create a tensor called B that contains 5 random uniforms with shape 1x1x7

# 4. Get the dot product of A and B after 2 elements of B have been randomly removed

# 5. Make y a tensor of scalar value 42 with gradient tracking.  Get the gradient of log(y) wrt y where y=42. (should be 1/42 = 0.0238)